# Logistic Regression — Model Experiments

This notebook runs all Logistic Regression experiments and logs them to the MLflow experiment **`LogisticRegression_Training`** on DagsHub.

Why Logistic Regression for fraud detection?
- Fast to train, easy to interpret coefficients.
- Sets a useful baseline that tree-based models must beat.
- Sensitive to scaling and to NaN handling — so it surfaces the *quality* of our cleaning + FE pipeline more clearly than tree models do.

We test several LR hyperparameter settings (different penalties, `C` values, `class_weight`) — these are all valid LR configurations; this notebook is purely about the **Logistic Regression classifier**.

## Notebook structure (required by the assignment)

1. Setup & MLflow Connection
2. Cleaning
3. Feature Engineering
4. Feature Selection
5. Training (incl. hyperparameter sweep + over-/under-fitting demos)
6. Final Pipeline & Logging

# 1. Setup & MLflow Connection

In [ ]:
# =============================================================
# Setup — auto-detects environment and locates the `src/` package.
#
# Resolution order:
#   1. Kaggle Dataset attached to this notebook that contains a `src/` folder
#      (recommended workflow while iterating, NO git push needed).
#   2. Fresh `git clone` of REPO_URL into /kaggle/working/repo
#      (use this once code is stable on GitHub).
#   3. Local Jupyter — notebook lives in <repo>/notebooks/, so `..` works.
#
# Set REPO_URL only if you intend to use option 2.
# =============================================================
import os, sys, subprocess, shutil

REPO_URL = "https://github.com/ekatsirekidze/ML_HW2.git"   # only used for option 2

ON_KAGGLE  = os.path.exists("/kaggle/input")
SRC_FOUND  = None

if ON_KAGGLE:
    # --- 1) Look for an attached Kaggle Dataset that contains a `src/` folder ---
    for _d in os.listdir("/kaggle/input"):
        _candidate = f"/kaggle/input/{_d}"
        if os.path.isdir(os.path.join(_candidate, "src")):
            SRC_FOUND = _candidate
            break

    if SRC_FOUND is None:
        # --- 2) fallback: git clone ---
        REPO_DIR = "/kaggle/working/repo"
        if os.path.isdir(REPO_DIR):
            shutil.rmtree(REPO_DIR)
        subprocess.check_call(["git", "clone", "-q", REPO_URL, REPO_DIR])
        SRC_FOUND = REPO_DIR

    sys.path.insert(0, SRC_FOUND)

    # --- install mlflow stack (Kaggle has everything else pre-installed) ---
    for _pkg in ("mlflow==2.10.2", "dagshub"):
        try:
            __import__(_pkg.split("==")[0].replace("-", "_"))
        except ImportError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _pkg])

    # --- load MLflow credentials from Kaggle Secrets ---
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    for _k in ("MLFLOW_TRACKING_URI", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"):
        os.environ[_k] = _s.get_secret(_k)

    print("src/ found at :", SRC_FOUND)
else:
    # --- 3) local Jupyter ---
    sys.path.insert(0, "..")

# --- shared imports (work the same in both environments) ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import StratifiedKFold, TimeSeriesSplit

from src.data import load_train, downcast_numerics
from src.preprocessing import IEEECleaner
from src.feature_engineering import IEEEFeatureEngineer
from src.feature_selection import (
    find_high_correlation, compute_mutual_info, ColumnSubsetSelector,
)
from src.mlflow_utils import (
    setup_mlflow, log_run, log_metrics_dict,
    evaluate_holdout, cross_validate_auc,
    plot_roc, plot_confusion, plot_feature_importance,
)
import mlflow, mlflow.sklearn

setup_mlflow("LogisticRegression_Training")
print("Environment :", "Kaggle" if ON_KAGGLE else "Local")
print("Tracking URI:", mlflow.get_tracking_uri())

## 1.1 Load data and create a time-based holdout

Because train and test in this competition are **temporally ordered**, the only honest holdout is "the last X% in time". We use the last 20% as the validation set; experiments below report `train_auc` (in-sample) and `val_auc` (this holdout). Cross-validation with `TimeSeriesSplit` is added later as a robustness check.

For Logistic Regression specifically, the full 590k×800+ matrix is slow to fit. Most exploratory runs use a **100k stratified subsample** so we iterate quickly; the *final* tuned model retrains on the full dataset.

In [ ]:
RANDOM_STATE  = 42
SUBSAMPLE     = 100_000        # used for fast exploration runs
HOLDOUT_FRAC  = 0.20           # last 20% (in time order) is the holdout
DATA_PATH = "/kaggle/input/competitions/ieee-fraud-detection"

df = downcast_numerics(load_train(path=DATA_PATH))
# df = downcast_numerics(load_train())
df = df.sort_values("TransactionDT").reset_index(drop=True)

split_idx = int(len(df) * (1 - HOLDOUT_FRAC))
train_full = df.iloc[:split_idx]
val_full   = df.iloc[split_idx:]

# Stratified subsample of the training side for fast experimentation
def subsample_stratified(d: pd.DataFrame, n: int, seed=RANDOM_STATE) -> pd.DataFrame:
    if len(d) <= n:
        return d
    pos = d[d["isFraud"] == 1]
    neg = d[d["isFraud"] == 0].sample(
        n - len(pos), random_state=seed
    ) if (n - len(pos)) < (d["isFraud"] == 0).sum() else d[d["isFraud"] == 0]
    return pd.concat([pos, neg]).sample(frac=1, random_state=seed).reset_index(drop=True)

train_small = subsample_stratified(train_full, SUBSAMPLE)

X_train_s, y_train_s = train_small.drop(columns=["isFraud"]), train_small["isFraud"]
X_val,     y_val     = val_full.drop(columns=["isFraud"]),    val_full["isFraud"]
X_train_f, y_train_f = train_full.drop(columns=["isFraud"]),  train_full["isFraud"]

print(f"train (full)  : {X_train_f.shape}  | fraud rate {y_train_f.mean():.3%}")
print(f"train (small) : {X_train_s.shape}  | fraud rate {y_train_s.mean():.3%}")
print(f"val           : {X_val.shape}  | fraud rate {y_val.mean():.3%}")

## 1.2 Pipeline factory

To keep every experiment fair, every run uses the same *shape* of pipeline — only the configuration of each step changes. The factory below takes config dicts for the four stages (cleaner / FE / preprocessor / model) and returns a fitted-or-not sklearn `Pipeline`.

For Logistic Regression we always need:
- **Numeric columns:** median imputation + `StandardScaler` (LR is scale-sensitive).
- **Categorical columns:** constant imputation + `OneHotEncoder(handle_unknown='ignore', max_categories=...)` to cap cardinality.

In [ ]:
def make_lr_pipeline(
    *,
    cleaner_kwargs: dict | None = None,
    fe_kwargs: dict | None = None,
    num_imputer: str = "median",         # 'median' | 'mean' | 'constant'
    num_imputer_const: float = 0.0,
    max_categories: int = 20,
    lr_kwargs: dict | None = None,
) -> Pipeline:
    """Build the full Logistic-Regression pipeline.

    The same factory is reused for every MLflow run in this notebook;
    runs differ only in the kwargs passed in.
    """
    cleaner_kwargs = cleaner_kwargs or {}
    fe_kwargs      = fe_kwargs      or {}
    lr_kwargs      = lr_kwargs      or {}

    if num_imputer == "constant":
        num_imp = SimpleImputer(strategy="constant", fill_value=num_imputer_const)
    else:
        num_imp = SimpleImputer(strategy=num_imputer)

    num_pipe = Pipeline([("impute", num_imp), ("scale", StandardScaler())])
    cat_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="missing")),
        ("ohe",    OneHotEncoder(handle_unknown="ignore", max_categories=max_categories,
                                 sparse_output=True)),
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", num_pipe, make_column_selector(dtype_include=np.number)),
            ("cat", cat_pipe, make_column_selector(dtype_include=object)),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )

    # Merge defaults with caller-supplied kwargs (caller always wins). This
    # avoids the "got multiple values for keyword argument" crash when a run
    # legitimately needs e.g. `max_iter=300` for the saga elasticnet config.
    # Also: `n_jobs` is only meaningful for the parallelisable solvers — we
    # skip it for liblinear, which would otherwise emit a UserWarning.
    user_lr  = dict(lr_kwargs)
    solver   = user_lr.get("solver", "lbfgs")
    defaults = {"max_iter": 200, "random_state": RANDOM_STATE}
    if solver in {"lbfgs", "newton-cg", "sag", "saga"}:
        defaults["n_jobs"] = -1
    final_lr_kwargs = {**defaults, **user_lr}

    return Pipeline([
        ("clean", IEEECleaner(**cleaner_kwargs)),
        ("fe",    IEEEFeatureEngineer(**fe_kwargs)),
        ("prep",  preprocessor),
        ("model", LogisticRegression(**final_lr_kwargs)),
    ])

# Smoke-test: build with defaults, ensure it instantiates
print(make_lr_pipeline())

# 2. Cleaning

We compare three strategies that vary the **NaN-fraction threshold** for column dropping and the **numeric imputation method**. Categorical NaNs are always filled with the literal `"missing"` (treated as its own level).

Each strategy is an MLflow run; we report `train_auc`, `val_auc`, `gap`. The best-by-`val_auc` strategy is carried forward into the FE stage.

| Run | missing_threshold | num imputer | rationale |
|---|---|---|---|
| `cleaning_v1_drop95_median`  | 0.95 | median   | safe baseline |
| `cleaning_v2_drop90_median`  | 0.90 | median   | aggressive drop |
| `cleaning_v3_drop99_const0`  | 0.99 | const(0) | keep nearly everything; constant impute |

In [ ]:
cleaning_grid = [
    dict(run="cleaning_v1_drop95_median",
         cleaner=dict(missing_threshold=0.95),
         num_imputer="median"),
    dict(run="cleaning_v2_drop90_median",
         cleaner=dict(missing_threshold=0.90),
         num_imputer="median"),
    dict(run="cleaning_v3_drop99_const0",
         cleaner=dict(missing_threshold=0.99),
         num_imputer="constant"),
]

# Use the small subsample for these runs — turn FE off so we measure CLEANING only.
cleaning_results = {}
for cfg in cleaning_grid:
    with log_run(
        cfg["run"],
        params={
            "missing_threshold": cfg["cleaner"]["missing_threshold"],
            "num_imputer":       cfg["num_imputer"],
            "subsample":         SUBSAMPLE,
            "fe_active":         False,
        },
        tags={
            "model_family": "LogisticRegression",
            "stage":        "cleaning",
        },
    ):
        pipe = make_lr_pipeline(
            cleaner_kwargs=cfg["cleaner"],
            fe_kwargs=dict(use_datetime=False, use_email=False,
                           use_freq=False, use_agg=False),
            num_imputer=cfg["num_imputer"],
            lr_kwargs=dict(C=1.0, penalty="l2", solver="liblinear"),
        )
        m = evaluate_holdout(pipe, X_train_s, y_train_s, X_val, y_val)
        log_metrics_dict(m)
        cleaning_results[cfg["run"]] = m
        print(f"{cfg['run']:35} train={m['train_auc']:.4f}  val={m['val_auc']:.4f}  gap={m['gap']:+.4f}")

best_cleaning = max(cleaning_results, key=lambda k: cleaning_results[k]["val_auc"])
print(f"\n>> best cleaning by val_auc: {best_cleaning}")

In [ ]:
# Pin the winning cleaning config — used as the base for FE/FS/Training stages.
BEST_CLEANING_KW   = next(c for c in cleaning_grid if c["run"] == best_cleaning)["cleaner"]
BEST_NUM_IMPUTER   = next(c for c in cleaning_grid if c["run"] == best_cleaning)["num_imputer"]
print("Locked in:", BEST_CLEANING_KW, "| num_imputer =", BEST_NUM_IMPUTER)

# 3. Feature Engineering

Now the cleaning step is fixed; we sweep the FE flags one block at a time, **cumulatively**. This isolates the effect of each block:

| Run | datetime | email | freq enc | aggregations |
|---|:-:|:-:|:-:|:-:|
| `fe_v1_baseline`     | – | – | – | – |
| `fe_v2_+datetime`    | ✓ | – | – | – |
| `fe_v3_+email`       | ✓ | ✓ | – | – |
| `fe_v4_+freq`        | ✓ | ✓ | ✓ | – |
| `fe_v5_full`         | ✓ | ✓ | ✓ | ✓ |

In [ ]:
fe_grid = [
    ("fe_v1_baseline",  dict(use_datetime=False, use_email=False, use_freq=False, use_agg=False)),
    ("fe_v2_+datetime", dict(use_datetime=True,  use_email=False, use_freq=False, use_agg=False)),
    ("fe_v3_+email",    dict(use_datetime=True,  use_email=True,  use_freq=False, use_agg=False)),
    ("fe_v4_+freq",     dict(use_datetime=True,  use_email=True,  use_freq=True,  use_agg=False)),
    ("fe_v5_full",      dict(use_datetime=True,  use_email=True,  use_freq=True,  use_agg=True)),
]

fe_results = {}
for run_name, fe_kw in fe_grid:
    with log_run(
        run_name,
        params={**fe_kw,
                "subsample": SUBSAMPLE,
                "cleaning":  best_cleaning},
        tags={"model_family": "LogisticRegression", "stage": "fe"},
    ):
        pipe = make_lr_pipeline(
            cleaner_kwargs=BEST_CLEANING_KW,
            fe_kwargs=fe_kw,
            num_imputer=BEST_NUM_IMPUTER,
            lr_kwargs=dict(C=1.0, penalty="l2", solver="liblinear"),
        )
        m = evaluate_holdout(pipe, X_train_s, y_train_s, X_val, y_val)
        log_metrics_dict(m)
        fe_results[run_name] = m
        print(f"{run_name:20} train={m['train_auc']:.4f}  val={m['val_auc']:.4f}  gap={m['gap']:+.4f}")

best_fe_name = max(fe_results, key=lambda k: fe_results[k]["val_auc"])
BEST_FE_KW   = dict(fe_grid)[best_fe_name]
print(f"\n>> best FE by val_auc: {best_fe_name} -> {BEST_FE_KW}")

# 4. Feature Selection

Three strategies, each applied **after** cleaning + FE:

1. **`fs_v1_all`** — keep everything (baseline).
2. **`fs_v2_drop_corr95`** — drop numeric features with pairwise corr > 0.95 (V columns are highly redundant).
3. **`fs_v3_top_mi200`** — keep the top 200 numeric features by mutual information.
4. **`fs_v4_l1_select`** — fit an L1-Logistic on a subsample and keep coefficients ≠ 0.

Selection is performed once on a *fitted* preview of the FE output, then the chosen feature list is locked into the pipeline via `ColumnSubsetSelector` for the eval run.

In [ ]:
# Materialise the cleaned + FE'd training subsample so we can analyse columns directly.
prep_view = Pipeline([
    ("clean", IEEECleaner(**BEST_CLEANING_KW)),
    ("fe",    IEEEFeatureEngineer(**BEST_FE_KW)),
]).fit(X_train_s, y_train_s).transform(X_train_s)

print(f"After cleaning + FE: shape = {prep_view.shape}")
prep_view.dtypes.value_counts()

In [ ]:
# --- Pre-compute the four feature lists at the SOURCE-column level ---
# (operating on `prep_view` = post-cleaning, post-FE training subsample)

cat_cols_src = prep_view.select_dtypes(include="object").columns.tolist()
num_cols_src = prep_view.select_dtypes(include=np.number).columns.tolist()

# v1: keep everything
all_cols   = prep_view.columns.tolist()

# v2: drop highly correlated numeric columns (>0.95)
high_corr  = find_high_correlation(prep_view[num_cols_src], threshold=0.95)
fs_v2_keep = [c for c in all_cols if c not in high_corr]

# v3: top 200 numeric features by mutual information; always keep all categoricals
mi = compute_mutual_info(prep_view[num_cols_src], y_train_s, sample=50_000)
fs_v3_keep = list(dict.fromkeys(mi.head(200).index.tolist() + cat_cols_src))

# v4: L1-Logistic selection on standardised numeric features only.
# Categoricals are kept unconditionally (the LR pipeline OHEs them anyway).
from sklearn.linear_model import LogisticRegression as _LR_
from sklearn.preprocessing import StandardScaler as _SS_

Xn_filled  = prep_view[num_cols_src].fillna(-999).values
Xn_scaled  = _SS_().fit_transform(Xn_filled)
_lr_l1     = _LR_(penalty="l1", C=0.1, solver="liblinear",
                  max_iter=300, random_state=RANDOM_STATE).fit(Xn_scaled, y_train_s)
nz_mask    = (_lr_l1.coef_[0] != 0)
fs_v4_keep = [c for c, keep in zip(num_cols_src, nz_mask) if keep] + cat_cols_src

print(f"v1_all          : {len(all_cols):4d}")
print(f"v2_drop_corr95  : {len(fs_v2_keep):4d}  (dropped {len(all_cols) - len(fs_v2_keep)})")
print(f"v3_top_mi200    : {len(fs_v3_keep):4d}")
print(f"v4_l1_select    : {len(fs_v4_keep):4d}  ({nz_mask.sum()}/{len(num_cols_src)} numeric kept)")

In [ ]:
def make_lr_pipeline_with_subset(keep_cols: list[str] | None, **kw) -> Pipeline:
    """Wrap make_lr_pipeline so we can prepend a ColumnSubsetSelector after FE."""
    base = make_lr_pipeline(**kw)
    if keep_cols is None:
        return base
    # Insert subset selector between FE and prep
    steps = list(base.steps)
    fe_idx = next(i for i, (n, _) in enumerate(steps) if n == "fe")
    steps.insert(fe_idx + 1, ("subset", ColumnSubsetSelector(keep_cols)))
    return Pipeline(steps)


fs_grid = [
    ("fs_v1_all",          None),
    ("fs_v2_drop_corr95",  fs_v2_keep),
    ("fs_v3_top_mi200",    fs_v3_keep),
    ("fs_v4_l1_select",    fs_v4_keep),
]

fs_results = {}
for run_name, keep in fs_grid:
    with log_run(
        run_name,
        params={"n_features_kept": (len(keep) if keep else len(all_cols)),
                "subsample": SUBSAMPLE,
                "cleaning":  best_cleaning,
                "fe":        best_fe_name},
        tags={"model_family": "LogisticRegression", "stage": "fs"},
    ):
        pipe = make_lr_pipeline_with_subset(
            keep_cols=keep,
            cleaner_kwargs=BEST_CLEANING_KW,
            fe_kwargs=BEST_FE_KW,
            num_imputer=BEST_NUM_IMPUTER,
            lr_kwargs=dict(C=1.0, penalty="l2", solver="liblinear"),
        )
        m = evaluate_holdout(pipe, X_train_s, y_train_s, X_val, y_val)
        log_metrics_dict(m)
        fs_results[run_name] = m
        print(f"{run_name:22} train={m['train_auc']:.4f}  val={m['val_auc']:.4f}  gap={m['gap']:+.4f}")

best_fs_name = max(fs_results, key=lambda k: fs_results[k]["val_auc"])
BEST_FS_KEEP = dict(fs_grid)[best_fs_name]
print(f"\n>> best FS by val_auc: {best_fs_name}")

# 5. Training

Now cleaning + FE + FS are locked in. We sweep Logistic Regression hyperparameters and explicitly include over/underfit demos.

| Run | penalty | C | class_weight | purpose |
|---|---|---|---|---|
| `train_v1_baseline_l2_C1`   | l2 | 1.0  | none      | baseline |
| `train_v2_l2_C0_01_under`   | l2 | 0.01 | none      | strong shrinkage → expected **underfit** |
| `train_v3_none_overfit`     | none | – | none      | no shrinkage → expected **overfit** (small gap because LR is high-bias by nature) |
| `train_v4_l1_C1`            | l1 | 1.0 | none      | sparsity |
| `train_v5_l2_balanced`      | l2 | 1.0 | balanced  | class-imbalance handling |
| `train_v6_elasticnet`       | elasticnet | 1.0 | none | combines L1+L2 |
| `train_v7_C_sweep`          | l2 | grid | none     | hyperparameter sweep (logged as nested runs) |

In [ ]:
import time
from sklearn.metrics import roc_auc_score

# --- Precompute the heavy cleaning + FE + OHE just ONCE ---------------------
# Section 5 only varies LR hyperparameters; cleaning / FE / FS / scaling /
# OHE are identical for every run, so re-running them inside every pipeline
# wastes most of the time. We build the full pipeline once, drop the LR
# step, fit on the training subsample, and transform both train + holdout
# to sparse matrices. Each subsequent experiment then fits just the
# `LogisticRegression` step on the already-transformed matrix — seconds
# instead of minutes per run, with identical metrics.
print("Precomputing transformed matrices for the LR training sweep ...")
_t0 = time.time()
_prep_only = make_lr_pipeline_with_subset(
    keep_cols=BEST_FS_KEEP,
    cleaner_kwargs=BEST_CLEANING_KW,
    fe_kwargs=BEST_FE_KW,
    num_imputer=BEST_NUM_IMPUTER,
    lr_kwargs=dict(penalty="l2", C=1.0, solver="liblinear"),
)
_prep_only.steps = _prep_only.steps[:-1]                  # drop the LR step
_prep_only.fit(X_train_s, y_train_s)
X_tr_mat  = _prep_only.transform(X_train_s)
X_val_mat = _prep_only.transform(X_val)
print(f"  done in {time.time()-_t0:.1f}s | train={X_tr_mat.shape} | val={X_val_mat.shape}")


def run_lr_training(run_name, lr_kwargs, *, purpose=None, extra_params=None):
    """Fit just the final LogisticRegression on the precomputed matrices.

    Logs the same params / metrics / plots to MLflow as the original
    pipeline-based version — the ONLY change is that we no longer redo
    cleaning + FE + OHE per run.
    """
    user_lr  = dict(lr_kwargs)
    solver   = user_lr.get("solver", "lbfgs")
    defaults = {"max_iter": 200, "random_state": RANDOM_STATE}
    if solver in {"lbfgs", "newton-cg", "sag", "saga"}:
        defaults["n_jobs"] = -1
    lr = LogisticRegression(**{**defaults, **user_lr})

    with log_run(
        run_name,
        params={**lr_kwargs,
                "subsample":      SUBSAMPLE,
                "cleaning":       best_cleaning,
                "fe":             best_fe_name,
                "feature_select": best_fs_name,
                **(extra_params or {})},
        tags={"model_family": "LogisticRegression",
              "stage": "training",
              **({"purpose": purpose} if purpose else {})},
    ):
        t0 = time.time()
        lr.fit(X_tr_mat, y_train_s)
        fit_sec = time.time() - t0
        p_tr  = lr.predict_proba(X_tr_mat)[:, 1]
        p_val = lr.predict_proba(X_val_mat)[:, 1]
        train_auc = float(roc_auc_score(y_train_s, p_tr))
        val_auc   = float(roc_auc_score(y_val,     p_val))
        m = {"train_auc": train_auc, "val_auc": val_auc,
             "gap": train_auc - val_auc, "fit_sec": float(fit_sec)}
        log_metrics_dict(m)

        mlflow.log_figure(plot_roc(y_val, p_val, title=run_name), "roc.png")
        mlflow.log_figure(plot_confusion(y_val, (p_val >= 0.5).astype(int),
                                         title=run_name), "confusion.png")

        print(f"{run_name:32} train={train_auc:.4f}  val={val_auc:.4f}  "
              f"gap={train_auc-val_auc:+.4f}  ({fit_sec:.1f}s)")
        return m


train_results = {}
train_results["v1"] = run_lr_training("train_v1_baseline_l2_C1",
    dict(penalty="l2", C=1.0, solver="liblinear"))
train_results["v2"] = run_lr_training("train_v2_l2_C0_01_under",
    dict(penalty="l2", C=0.01, solver="liblinear"),
    purpose="underfit_demo")
train_results["v3"] = run_lr_training("train_v3_none_overfit",
    dict(penalty=None, solver="lbfgs"),
    purpose="overfit_demo")
train_results["v4"] = run_lr_training("train_v4_l1_C1",
    dict(penalty="l1", C=1.0, solver="liblinear"))
train_results["v5"] = run_lr_training("train_v5_l2_balanced",
    dict(penalty="l2", C=1.0, solver="liblinear", class_weight="balanced"))
train_results["v6"] = run_lr_training("train_v6_elasticnet",
    dict(penalty="elasticnet", C=1.0, l1_ratio=0.5, solver="saga", max_iter=300))

In [ ]:
# --- C sweep, logged as a parent run with nested children -----------------
# Reuses the X_tr_mat / X_val_mat precomputed in the previous cell, so each
# child run fits in seconds instead of redoing cleaning + FE + OHE.
C_grid = [0.01, 0.1, 0.5, 1.0, 2.0, 5.0]
sweep_rows = []

with log_run(
    "train_v7_C_sweep_parent",
    params={"penalty": "l2", "solver": "liblinear",
            "C_grid": str(C_grid), "subsample": SUBSAMPLE},
    tags={"model_family": "LogisticRegression", "stage": "hpo"},
):
    for C in C_grid:
        with log_run(
            f"train_v7_C_sweep__C={C}",
            params={"penalty": "l2", "C": C, "solver": "liblinear"},
            tags={"model_family": "LogisticRegression",
                  "stage": "hpo", "parent": "C_sweep"},
            nested=True,
        ):
            lr = LogisticRegression(
                penalty="l2", C=C, solver="liblinear",
                max_iter=200, random_state=RANDOM_STATE,
            )
            t0 = time.time()
            lr.fit(X_tr_mat, y_train_s)
            fit_sec = time.time() - t0
            p_tr  = lr.predict_proba(X_tr_mat)[:, 1]
            p_val = lr.predict_proba(X_val_mat)[:, 1]
            tr_auc  = float(roc_auc_score(y_train_s, p_tr))
            val_auc = float(roc_auc_score(y_val,     p_val))
            m = {"train_auc": tr_auc, "val_auc": val_auc,
                 "gap": tr_auc - val_auc, "fit_sec": float(fit_sec)}
            log_metrics_dict(m)
            sweep_rows.append({"C": C, **m})
            print(f"  C={C:>5}  train={tr_auc:.4f}  val={val_auc:.4f}  "
                  f"gap={tr_auc-val_auc:+.4f}  ({fit_sec:.1f}s)")

sweep_df = pd.DataFrame(sweep_rows).sort_values("val_auc", ascending=False)
sweep_df

# 6. Final Pipeline & Logging

We pick the best `(penalty, C)` from the sweep and refit on the **full** training set. Then we:

1. Cross-validate it with both `StratifiedKFold(5)` and `TimeSeriesSplit(5)` and log per-fold AUCs. Comparing the two reveals temporal leakage in random K-Fold.
2. Re-fit on `train_full` and report the holdout AUC on `val_full`.
3. Save the entire `Pipeline` (cleaner → FE → preprocessor → LR) via `mlflow.sklearn.log_model(...)` and register it as **`LogisticRegression_Fraud_Pipeline`** in the Model Registry.

The registered model can be `predict_proba`-ed on raw, un-preprocessed test data — the pipeline does all the cleaning + FE itself.

In [ ]:
# Pick the best HPO configuration
best_C = float(sweep_df.iloc[0]["C"])
print("Final config: penalty=l2, C =", best_C)

final_pipe = make_lr_pipeline_with_subset(
    keep_cols=BEST_FS_KEEP,
    cleaner_kwargs=BEST_CLEANING_KW,
    fe_kwargs=BEST_FE_KW,
    num_imputer=BEST_NUM_IMPUTER,
    lr_kwargs=dict(penalty="l2", C=best_C, solver="liblinear"),
)

In [ ]:
# --- Cross-validation comparison: StratifiedKFold vs TimeSeriesSplit ---
# (run on the SMALL sample to stay fast; the gap between the two is the signal)

with log_run(
    "cv_v1_stratified_5fold",
    params={"cv": "StratifiedKFold", "n_splits": 5,
            "C": best_C, "subsample": SUBSAMPLE},
    tags={"model_family": "LogisticRegression", "stage": "cv"},
):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    m = cross_validate_auc(final_pipe, X_train_s, y_train_s, cv=skf)
    log_metrics_dict(m)
    print("Stratified KFold:  val_auc =", m["val_auc"], "± ", m["val_auc_std"])

with log_run(
    "cv_v2_timeseries_5fold",
    params={"cv": "TimeSeriesSplit", "n_splits": 5,
            "C": best_C, "subsample": SUBSAMPLE,
            "note": "data already sorted by TransactionDT"},
    tags={"model_family": "LogisticRegression", "stage": "cv"},
):
    tss = TimeSeriesSplit(n_splits=5)
    m = cross_validate_auc(final_pipe, X_train_s, y_train_s, cv=tss)
    log_metrics_dict(m)
    print("TimeSeriesSplit :  val_auc =", m["val_auc"], "± ", m["val_auc_std"])

In [ ]:
# --- Final fit on FULL training data, evaluate on holdout, register the model ---

with log_run(
    "final_pipeline_full_train",
    params={
        "penalty": "l2", "C": best_C, "solver": "liblinear",
        "cleaning":       best_cleaning,
        "fe":             best_fe_name,
        "feature_select": best_fs_name,
        "trained_on":     "full_train (no subsample)",
    },
    tags={"model_family": "LogisticRegression",
          "stage":        "final",
          "purpose":      "register_in_model_registry"},
):
    m = evaluate_holdout(final_pipe, X_train_f, y_train_f, X_val, y_val)
    log_metrics_dict(m)

    proba = final_pipe.predict_proba(X_val)[:, 1]
    mlflow.log_figure(plot_roc(y_val, proba, title="Final LR — holdout ROC"),
                      "roc_final.png")
    mlflow.log_figure(plot_confusion(y_val, (proba >= 0.5).astype(int),
                                     title="Final LR — confusion @ 0.5"),
                      "confusion_final.png")

    # Log + register the entire Pipeline (raw input -> probability)
    mlflow.sklearn.log_model(
        sk_model=final_pipe,
        artifact_path="pipeline",
        registered_model_name="LogisticRegression_Fraud_Pipeline",
    )
    print(f"Registered model.  holdout val_auc = {m['val_auc']:.4f}  gap = {m['gap']:+.4f}")

## What we learned (write this up in the README)

After the runs above, paste the actual numbers from MLflow into the README:

- **Best cleaning:** `{best_cleaning}` — note the `val_auc` it achieved.
- **Best FE:** `{best_fe_name}` — adding which block helped most?
- **Best FS:** `{best_fs_name}` — how many features did the winner keep?
- **Underfit demo:** `train_v2_l2_C0_01_under` — both train and val AUC are low (high bias).
- **Overfit demo:** `train_v3_none_overfit` — train AUC clearly higher than val AUC (high variance).
- **CV gap signal:** `TimeSeriesSplit` AUC is *lower* than random `StratifiedKFold` — that's the temporal-leakage delta you must mention in the README.
- **Final holdout AUC:** record it here for direct comparison with the other 7 models.